# 🧠 Huấn luyện Random Forest 5 lớp (Layer 3/4)

**Pipeline chống overfitting:**
- Không dùng dữ liệu synthetic giả lập
- Đánh giá trên 15 raw features (không group features) để tránh data leakage
- Production model được train kèm group features (Controller tính realtime)
- Sử dụng `class_weight='balanced'` + regularization (`max_depth`, `min_samples_leaf`)

| Nhãn | Lớp | Mô tả |
|------|------|--------|
| 0 | BENIGN | Lưu lượng sạch |
| 1 | Brute Force | Dò mật khẩu SSH/FTP |
| 2 | DDoS | TCP/UDP/ICMP Flood |
| 3 | DoS | SYN Flood |
| 4 | Port Scan | Nmap quét cổng |

## Bước 1: Import thư viện

In [1]:
import pandas as pd
import numpy as np
import os, joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

## Bước 2: Nạp dữ liệu và gán nhãn

In [2]:
files = {
    'benign': ('benign.csv', 0),
    'bruteforce': ('bruteforce.csv', 1),
    'ddos': ('ddos.csv', 2),
    'dos': ('dos.csv', 3),
    'portscan': ('portscan.csv', 4)
}

benign_dfs = []
attack_dfs = {i: [] for i in range(1, 5)}

for name, (fname, label) in files.items():
    if not os.path.exists(fname):
        print(f'Skipping {fname}')
        continue
    print(f'✓ Loading {fname}...')
    df = pd.read_csv(fname)
    if name == 'benign':
        df['label'] = 0
        benign_dfs.append(df)
    else:
        attack_ips = {'10.0.0.8', '10.0.0.7'}
        if label == 2:
            attack_ips.add('10.0.0.6')
        is_attack = df['ip_src'].isin(attack_ips)
        df_atk = df[is_attack].copy()
        df_atk['label'] = label
        attack_dfs[label].append(df_atk)
        df_bg = df[~is_attack].copy()
        df_bg['label'] = 0
        benign_dfs.append(df_bg)

final_dfs = [pd.concat(benign_dfs, ignore_index=True)]
for lbl in range(1, 5):
    if attack_dfs[lbl]:
        final_dfs.append(pd.concat(attack_dfs[lbl], ignore_index=True))

df_all = pd.concat(final_dfs, ignore_index=True)
print(f'\nTổng mẫu: {len(df_all):,}')
print(df_all['label'].value_counts().sort_index())

✓ Loading benign.csv...
✓ Loading bruteforce.csv...
✓ Loading ddos.csv...


✓ Loading dos.csv...
✓ Loading portscan.csv...



Tổng mẫu: 756,946
label
0    367139
1       960
2    165343
3     83369
4    140135
Name: count, dtype: int64


## Bước 3: Chuẩn bị features và chia tập

Sử dụng 15 raw features từ OpenFlow flow stats. **Không dùng group features** trong đánh giá
để tránh data leakage (group features phụ thuộc vào các dòng cùng time window).

In [3]:
cols_to_drop = ['timestamp','datapath_id','flow_id','ip_src','ip_dst','tp_src','label']
X = df_all.drop(columns=cols_to_drop, errors='ignore')
y = df_all['label']

X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
print(f'Features ({X.shape[1]}): {list(X.columns)}')
print(f'Tổng mẫu: {X.shape[0]:,}')

Features (15): ['tp_dst', 'ip_proto', 'icmp_code', 'icmp_type', 'flow_duration_sec', 'flow_duration_nsec', 'idle_timeout', 'hard_timeout', 'flags', 'packet_count', 'byte_count', 'packet_count_per_second', 'packet_count_per_nsecond', 'byte_count_per_second', 'byte_count_per_nsecond']
Tổng mẫu: 756,946


In [4]:
# Chia 70/10/20
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.125, random_state=42, stratify=y_train_val
)

print(f'Train: {X_train.shape} ({len(X_train)/len(X)*100:.1f}%)')
print(f'Val  : {X_val.shape} ({len(X_val)/len(X)*100:.1f}%)')
print(f'Test : {X_test.shape} ({len(X_test)/len(X)*100:.1f}%)')

Train: (529861, 15) (70.0%)
Val  : (75695, 15) (10.0%)
Test : (151390, 15) (20.0%)


## Bước 4: Huấn luyện Random Forest

- `class_weight='balanced'`: tự cân bằng trọng số theo tỷ lệ lớp
- `max_depth=15, min_samples_leaf=10`: regularization chống overfitting

In [5]:
target_names = ['BENIGN','Brute Force','DDoS','DoS','Port Scan']

clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
clf.fit(X_train, y_train)
print('✓ Huấn luyện hoàn tất!')

✓ Huấn luyện hoàn tất!


## Bước 5: Đánh giá trên tập Validation

In [6]:
y_val_pred = clf.predict(X_val)
print(f'🎯 Validation Accuracy: {accuracy_score(y_val, y_val_pred)*100:.2f}%')
print('\n📋 Validation Classification Report:')
print(classification_report(y_val, y_val_pred, labels=range(5), target_names=target_names, zero_division=0))

🎯 Validation Accuracy: 94.78%

📋 Validation Classification Report:
              precision    recall  f1-score   support

      BENIGN       1.00      0.96      0.98     36714
 Brute Force       0.41      1.00      0.59        96
        DDoS       0.86      0.99      0.92     16534
         DoS       0.87      0.82      0.85      8337
   Port Scan       1.00      0.94      0.97     14014

    accuracy                           0.95     75695
   macro avg       0.83      0.94      0.86     75695
weighted avg       0.95      0.95      0.95     75695



## Bước 6: Đánh giá trên tập Test

In [7]:
y_test_pred = clf.predict(X_test)
print(f'🎯 Test Accuracy: {accuracy_score(y_test, y_test_pred)*100:.2f}%')
print('\n📊 Confusion Matrix (Test):')
print(confusion_matrix(y_test, y_test_pred))
print('\n📋 Test Classification Report:')
print(classification_report(y_test, y_test_pred, labels=range(5), target_names=target_names, zero_division=0))

🎯 Test Accuracy: 94.83%

📊 Confusion Matrix (Test):
[[70882    89  2402    39    16]
 [    0   192     0     0     0]
 [   33   130 32594   274    38]
 [    5    33  2996 13589    51]
 [    6    12    13  1692 26304]]

📋 Test Classification Report:


              precision    recall  f1-score   support

      BENIGN       1.00      0.97      0.98     73428
 Brute Force       0.42      1.00      0.59       192
        DDoS       0.86      0.99      0.92     33069
         DoS       0.87      0.81      0.84     16674
   Port Scan       1.00      0.94      0.97     28027

    accuracy                           0.95    151390
   macro avg       0.83      0.94      0.86    151390
weighted avg       0.95      0.95      0.95    151390



## Bước 7: Kiểm tra Overfitting

In [8]:
y_train_pred = clf.predict(X_train)
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)
print(f'Train Accuracy: {train_acc*100:.2f}%')
print(f'Test  Accuracy: {test_acc*100:.2f}%')
print(f'Gap           : {(train_acc - test_acc)*100:.2f}%')
if (train_acc - test_acc) < 0.02:
    print('✅ Gap < 2%: Mô hình không overfitting.')
else:
    print('⚠️  Gap >= 2%: Cần xem xét regularization mạnh hơn.')

Train Accuracy: 94.80%
Test  Accuracy: 94.83%
Gap           : -0.03%
✅ Gap < 2%: Mô hình không overfitting.


## Bước 8: 5-Fold Cross-Validation

In [9]:
cv_scores = cross_val_score(
    RandomForestClassifier(n_estimators=50, max_depth=15, min_samples_leaf=10,
                           class_weight='balanced', random_state=42, n_jobs=-1),
    X, y, cv=5, scoring='accuracy', n_jobs=-1
)
print(f'CV Scores: {cv_scores}')
print(f'CV Mean  : {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%')

CV Scores: [0.62741925 0.97845286 0.97935781 0.97516332 0.96777177]
CV Mean  : 90.56% ± 13.92%


## Bước 9: Feature Importance

In [10]:
importances = pd.DataFrame({
    'feature': clf.feature_names_in_,
    'importance': clf.feature_importances_
}).sort_values('importance', ascending=False)
print(importances.to_string(index=False))

                 feature  importance
                  tp_dst    0.318710
              byte_count    0.215988
            packet_count    0.145989
   byte_count_per_second    0.084743
                ip_proto    0.079815
      flow_duration_nsec    0.077456
 packet_count_per_second    0.050279
       flow_duration_sec    0.022866
  byte_count_per_nsecond    0.002883
               icmp_code    0.000727
               icmp_type    0.000469
packet_count_per_nsecond    0.000074
            idle_timeout    0.000000
            hard_timeout    0.000000
                   flags    0.000000


## Bước 10: Xuất mô hình Production

Production model sử dụng thêm **group features** vì Controller tính chúng realtime.
Model này train trên toàn bộ dữ liệu để tối đa hiệu suất triển khai.

In [11]:
def recalculate_group_features(df):
    df = df.copy()
    df['round_time'] = df['timestamp'].round(0)
    gp = df.groupby(['round_time', 'ip_src'])
    flow_count = gp.size().to_dict()
    packet_rate = gp['packet_count_per_second'].sum().to_dict()
    unique_ports = gp['tp_dst'].nunique().to_dict()
    keys = list(zip(df['round_time'], df['ip_src']))
    df['flow_count_src'] = [flow_count.get(k, 0) for k in keys]
    df['packet_rate_src'] = [packet_rate.get(k, 0.0) for k in keys]
    df['unique_dst_ports_src'] = [unique_ports.get(k, 0) for k in keys]
    return df.drop(columns=['round_time'])

df_prod = recalculate_group_features(df_all)
cols_drop_prod = ['timestamp','datapath_id','flow_id','ip_src','ip_dst','tp_src','label']
X_prod = df_prod.drop(columns=cols_drop_prod, errors='ignore').replace([np.inf,-np.inf], np.nan).fillna(0)
y_prod = df_prod['label']

clf_prod = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
clf_prod.fit(X_prod, y_prod)

model_path = 'rf_model_multiclass.pkl'
joblib.dump(clf_prod, model_path)
print(f'[+] Đã xuất mô hình production: {model_path}')
print(f'    Features ({len(clf_prod.feature_names_in_)}): {list(clf_prod.feature_names_in_)}')

[+] Đã xuất mô hình production: rf_model_multiclass.pkl
    Features (18): ['tp_dst', 'ip_proto', 'icmp_code', 'icmp_type', 'flow_duration_sec', 'flow_duration_nsec', 'idle_timeout', 'hard_timeout', 'flags', 'packet_count', 'byte_count', 'packet_count_per_second', 'packet_count_per_nsecond', 'byte_count_per_second', 'byte_count_per_nsecond', 'flow_count_src', 'packet_rate_src', 'unique_dst_ports_src']
